# CatBoost Model Training 

## Input file (place in the same folder)
- `ml_train_dataset.xlsx`

## Outputs
- `ml/final_catboost_model.cbm`
- `ml/final_catboost_model.meta.json`

In [ ]:
!pip install -r requirements.txt >nul 2>&1

In [ ]:

import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, recall_score, f1_score, accuracy_score
import optuna, json
from pathlib import Path

INPUT_XLSX = 'ml_train_dataset.xlsx'  
MODEL_OUT  = 'final_catboost_model.cbm'
META_OUT   = 'final_catboost_model.meta.json'

# Features and target 
features = ['FBO type','Product type','Operation type','Former Inspection result','FBO Size','Inspection interval (days)']
cat_features = ['FBO type','Product type','Operation type','Former Inspection result','FBO Size']  # categorical (even though encoded as ints)
print('Loading:', INPUT_XLSX)
df = pd.read_excel(INPUT_XLSX)

# Column presence check 
required_cols = features + ["Behavior Change"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns in {INPUT_XLSX}: {missing}")

# Explicit target mapping + validation to avoid silent label corruption (e.g., NaN/unknown values).
raw_to_model = {-3: 0, -2: 0, -1: 0, 0: 1, 1: 2, 2: 2, 3: 2}
raw_target = df["Behavior Change"]
y = raw_target.map(raw_to_model)
if y.isna().any():
    bad_vals = raw_target[y.isna()].unique()
    raise ValueError(f"Unexpected values in 'Behavior Change': {bad_vals}")
y = y.astype(int)

X = df[features].copy()  # already encoded to ints

print("Target counts:", y.value_counts().sort_index().to_dict())

# Train-val-test split 
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# Class weights 
classes = sorted(y.unique())
train_counts = y_train.value_counts()
total = len(y_train)
class_weights = []
for c in classes:
    if c not in train_counts:
        raise ValueError(f"Class {c} missing in training split; cannot compute class weights.")
    class_weights.append(total / (len(classes) * train_counts[c]))
print("Class weights:", class_weights)

# Use categorical feature indices for Pool 
cat_feature_indices = [features.index(c) for c in cat_features]

# CatBoost 
train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
val_pool   = Pool(X_val,   y_val,   cat_features=cat_feature_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_feature_indices)


# Optuna objective
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_seed': 42,
        'loss_function': 'MultiClass',
        'eval_metric': 'MultiClass',
        'class_weights': class_weights,
        'early_stopping_rounds': 50,
        'verbose': False,
    }
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    preds = model.predict(val_pool).astype(int).ravel()
    return f1_score(y_val, preds, average='macro')

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print("Best params:", study.best_trial.params)

# Final model
best_params = study.best_trial.params
best_params.update({
    'loss_function': 'MultiClass',
    'class_weights': class_weights,
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'verbose': 100,
})
final_model = CatBoostClassifier(**best_params)
final_model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# Eval
val_preds = final_model.predict(val_pool).astype(int).ravel()
print("Validation report:\n", classification_report(y_val, val_preds, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_val, val_preds))

test_preds = final_model.predict(test_pool).astype(int).ravel()


print("Test report:\n", classification_report(y_test, test_preds, zero_division=0))
print("Test confusion matrix:\n", confusion_matrix(y_test, test_preds))
print("Test macro-F1:", f1_score(y_test, test_preds, average='macro'))
print("Test accuracy:", accuracy_score(y_test, test_preds))

# Save model + meta for ABM
final_model.save_model(MODEL_OUT)
meta = {
    'feature_order': features,            # ABM will follow this order
    'categorical_features': cat_features, # info only
    'target_mapping': {0:-1, 1:0, 2:1},   # model → ABM mapping
}
Path(META_OUT).write_text(json.dumps(meta, indent=2))
print("Saved:\n ", MODEL_OUT, "\n ", META_OUT)
